## Import

In [3]:
!pip install imbalanced-learn nrclex

Defaulting to user installation because normal site-packages is not writeable


In [1]:
from scipy import sparse
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from imblearn.over_sampling import SMOTE
from collections import Counter
from itertools import product
import numpy as np
import operator
import nltk
import math
from scipy.stats import norm
import string
from tqdm import tqdm
from sklearn.metrics import f1_score

In [2]:
!python -m nltk.downloader punkt

<frozen runpy>:128: RuntimeWarning: 'nltk.downloader' found in sys.modules after import of package 'nltk', but prior to execution of 'nltk.downloader'; this may result in unpredictable behaviour
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\great\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## File Loading

In [2]:
def load_data(filename):
    X = []
    Y = []
    i = 0
    with open(filename, encoding="utf-8") as file:
        for line in file:
            if i == 0:
                i += 1
                continue
            cols = line.split("\t")
            idd = cols[0]
            label = cols[1].lstrip().rstrip()
            text = cols[2].lstrip().rstrip()

            X.append(text)
            Y.append(label)

    return X, Y


## Classifier Class

In [3]:
class Classifier:
    def __init__(self, feature_methods: list, trainX, trainY, devX, devY, testX, testY):
        self.feature_vocab = {}
        self.feature_methods = feature_methods
        self.min_feature_count=2
        self.log_reg = None

        self.trainY=trainY
        self.devY=devY
        self.testY=testY
        
        self.trainX = self.process(trainX, training=True)
        self.devX = self.process(devX, training=False)
        self.testX = self.process(testX, training=False)

    # Featurize entire dataset
    def featurize(self, data):
        featurized_data = []
        for text in data:
            combined_features = {}
            for feature_method in self.feature_methods:
                feats = feature_method(text)
                combined_features.update(feats)  # Merge dictionaries
            featurized_data.append(combined_features)
        return featurized_data

    # Read dataset and returned featurized representation as sparse matrix + label array
    def process(self, X_data, training = False):
        data = self.featurize(X_data)

        if training:
            fid = 0
            feature_doc_count = Counter()
            for feats in data:
                for feat in feats.keys():  # Use keys of the dictionary
                    feature_doc_count[feat] += 1

            for feat in feature_doc_count:
                if feature_doc_count[feat] >= self.min_feature_count:
                    self.feature_vocab[feat] = fid
                    fid += 1

        F = len(self.feature_vocab)
        D = len(data)
        X = sparse.dok_matrix((D, F))
        for idx, feats in enumerate(data):
            for feat, value in feats.items():  # Use key-value pairs
                if feat in self.feature_vocab:
                    X[idx, self.feature_vocab[feat]] = value

        return X

    # Train model and evaluate on held-out data
    def train(self):
        min_class_count = min(Counter(self.trainY).values())
        k_neighbors = min(5, min_class_count - 1)

        if k_neighbors >= 1:
            smote = SMOTE(k_neighbors=k_neighbors)
            trainX_resampled, trainY_resampled = smote.fit_resample(self.trainX, self.trainY)
        else:
            trainX_resampled, trainY_resampled = self.trainX, self.trainY

        C_vals = [0.01, 0.1, 1, 10, 100]
        # Valid combinations: l1 only works with squared_hinge
        param_grid = [
            (C, penalty, loss)
            for C, penalty, loss in product(C_vals, ['l1', 'l2'], ['hinge', 'squared_hinge'])
            if not (penalty == 'l1' and loss == 'hinge')
        ]

        best_dev_accuracy=0
        best_model=None
        for C, penalty, loss in param_grid:
            self.log_reg = OneVsRestClassifier(
                LinearSVC(C=C, penalty=penalty, loss=loss, max_iter=1000000, class_weight='balanced')
            )
            self.log_reg.fit(trainX_resampled, trainY_resampled)
            development_accuracy = self.log_reg.score(self.devX, self.devY)
            if development_accuracy > best_dev_accuracy:
                best_dev_accuracy=development_accuracy
                best_model=self.log_reg

        self.log_reg=best_model
        

    def test(self):
        return self.log_reg.score(self.testX, self.testY)
    
    def f1_test(self):
        predictions = self.log_reg.predict(self.testX)
        return f1_score(self.testY, predictions, average='weighted'), f1_score(self.testY, predictions, average="macro")

    def printWeights(self, n=10):
        reverse_vocab=[None]*len(self.log_reg.estimators_[0].coef_[0])
        for k in self.feature_vocab:
            reverse_vocab[self.feature_vocab[k]]=k

        for i, cat in enumerate(self.log_reg.classes_):
            weights = self.log_reg.estimators_[i].coef_[0]
            for feature, weight in list(reversed(sorted(zip(reverse_vocab, weights), key=operator.itemgetter(1))))[:n]:
                print("%s\t%.3f\t%s" % (cat, weight, feature))
            print()

## Feature Engineering

In [4]:
X, Y = load_data("../adjuticated.txt")
stopwords = set(nltk.corpus.stopwords.words('english'))
punctuation = set(string.punctuation)
punctuation.add("``")
punctuation.add("''")
punctuation.add('’')
punctuation.add('“')
punctuation.add('”')
punctuation.add('—')
punctuation.add("'s")
words_fear = {}
words_name_calling = {}
for text, label in zip(X, Y):
    if label == "Appeal to Fear":
        for word in nltk.word_tokenize(text):
            word=word.lower()
            if word not in stopwords and word not in punctuation:
                if word not in words_fear.keys():
                    words_fear[word]=0
                words_fear[word]+=1
    elif label == "Name Calling":
        for word in nltk.word_tokenize(text):
            word=word.lower()
            if word not in stopwords and word not in punctuation:
                if word not in words_name_calling.keys():
                    words_name_calling[word]=0
                words_name_calling[word]+=1

words_fear = sorted(words_fear.items(), key=operator.itemgetter(1), reverse=True)
words_name_calling = sorted(words_name_calling.items(), key=operator.itemgetter(1), reverse=True)

print("Appeal to Fear:", words_fear[:15], "\nName Calling:", words_name_calling[:15])
print("Classes:", set(Y))


Appeal to Fear: [('act', 23), ('security', 22), ('must', 20), ('congress', 16), ('also', 15), ('nation', 12), ('national', 11), ('health', 11), ('social', 11), ('would', 10), ('ensure', 10), ('tax', 10), ('access', 9), ('care', 9), ('legislation', 9)] 
Name Calling: [('government', 11), ('act', 10), ('federal', 10), ('energy', 9), ('congress', 8), ('president', 8), ('trump', 8), ('funding', 8), ('tax', 7), ('would', 6), ('work', 6), ('time', 6), ('less', 5), ('americans', 5), ('benefits', 5)]
Classes: {'Appeal to Fear', 'Loaded Language', 'Name Calling', 'No Propaganda'}


In [5]:
def binary_bow_featurize(text):
    feats = {}
    words = nltk.word_tokenize(text)

    for word in words:
        word=word.lower()
        feats[word]=1
            
    return feats

def bigram(text):
    feats = {}
    tokens = [w.lower() for w in nltk.word_tokenize(text) if w.isalpha()]
    for i in range(len(tokens) - 1):
        feats[f"{tokens[i]}_{tokens[i+1]}"] = 1

    return feats

def trigram(text):
    feats = {}
    tokens = [w.lower() for w in nltk.word_tokenize(text) if w.isalpha()]
    for i in range(len(tokens) - 2):
        feats[f"{tokens[i]}_{tokens[i+1]}_{tokens[i+2]}"] = 1

    return feats

def quadrigram(text):
    feats = {}
    tokens = [w.lower() for w in nltk.word_tokenize(text) if w.isalpha()]
    for i in range(len(tokens) - 3):
        feats[f"{tokens[i]}_{tokens[i+1]}_{tokens[i+2]}_{tokens[i+3]}"] = 1

    return feats

def appeal_to_fear_featurize(text):
    feats = {}
    words = nltk.word_tokenize(text)

    for word in words:
        word=word.lower()
        if word in ["act", "security", "must", "congress", "also", "nation", "national", "health", 
                    "social", "tax", "access", "care", "legislation"]:
            feats[word] = feats.get(word, 0) + 1
            
    return feats

def name_calling_featurize(text):
    feats = {}
    words = nltk.word_tokenize(text)

    for word in words:
        word=word.lower()
        if word in ["government", "act", "federal", "energy", "congress", "president", 
                    "trump", "funding", "tax", "americans", "benefits"]:
            feats[word] = feats.get(word, 0) + 1
            
    return feats

def sentence_structure_featurize(text):
    feats = {}
    sentences = nltk.sent_tokenize(text)
    feats['num_sentences'] = len(sentences)
    words = nltk.word_tokenize(text)
    feats['num_words'] = len(words)
    feats['avg_sentence_length'] = len(words) / (len(sentences) or 1)
    return feats

def pronoun_featurize(text):
    words = [w.lower() for w in nltk.word_tokenize(text)]
    total = (len(words) or 1)
    feats = {}
    feats['first_person_pronoun_ratio'] = sum(w in ['i', 'me', 'my', 'mine', 'we', 'us', 'our', 'ours'] for w in words) / total
    feats['second_person_pronoun_ratio'] = sum(w in ['you', 'your', 'yours'] for w in words) / total
    feats['third_person_pronoun_ratio'] = sum(w in ['he', 'him', 'his', 'she', 'her', 'hers', 
                                                    'they', 'them', 'their', 'theirs'] for w in words) / total
    return feats

In [6]:
def confidence_intervals(accuracy, n, significance_level):
    critical_value=(1-significance_level)/2
    z_alpha=-1*norm.ppf(critical_value)
    se=math.sqrt((accuracy*(1-accuracy))/n)
    return accuracy-(se*z_alpha), accuracy+(se*z_alpha)

## Run Function

In [7]:
def run(trainingFile, devFile, testFile):
    trainX, trainY=load_data(trainingFile)
    devX, devY=load_data(devFile)
    testX, testY=load_data(testFile)

    for feature in tqdm([binary_bow_featurize, bigram, trigram, appeal_to_fear_featurize, name_calling_featurize, sentence_structure_featurize, pronoun_featurize]):
        classifier = Classifier([feature], trainX, trainY, devX, devY, testX, testY)
        classifier.train()
        accuracy = classifier.test()
        f1_weighted, f1_macro = classifier.f1_test()
        lower, upper=confidence_intervals(accuracy, len(testY), .95)
        weighted_f1_lower, weighted_f1_upper = confidence_intervals(f1_weighted, len(testY), .95)
        macro_f1_lower, macro_f1_upper = confidence_intervals(f1_macro, len(testY), .95)
        print(f"Test accuracy for '{feature.__name__}': {accuracy:.5f}, 95% CIs for accuracy: [{lower:.5f} {upper:.5f}]")
        print(f"Test F1 scores for '{feature.__name__}': \
              Weighted: {f1_weighted:.5f} [{weighted_f1_lower:.5f}, {weighted_f1_upper:.5f}], \
              Macro: {f1_macro:.5f} [{macro_f1_lower:.5f}, {macro_f1_upper:.5f}]\n")

    # simple_classifier.printWeights()

In [8]:
import warnings
warnings.filterwarnings("ignore")
trainingFile = "splits/train.txt"
devFile = "splits/dev.txt"
testFile = "splits/test.txt"
run(trainingFile, devFile, testFile)

 14%|█▍        | 1/7 [00:14<01:28, 14.73s/it]

Test accuracy for 'binary_bow_featurize': 0.57000, 95% CIs for accuracy: [0.47297 0.66703]
Test F1 scores for 'binary_bow_featurize':               Weighted: 0.55198 [0.45451, 0.64944],               Macro: 0.36413 [0.26982, 0.45844]



 29%|██▊       | 2/7 [00:21<00:50, 10.12s/it]

Test accuracy for 'bigram': 0.59000, 95% CIs for accuracy: [0.49360 0.68640]
Test F1 scores for 'bigram':               Weighted: 0.56485 [0.46768, 0.66202],               Macro: 0.39182 [0.29614, 0.48750]



 43%|████▎     | 3/7 [00:24<00:26,  6.65s/it]

...

Test accuracy for 'name_calling_featurize': 0.47000, 95% CIs for accuracy: [0.37218 0.56782]
Test F1 scores for 'name_calling_featurize':               Weighted: 0.30054 [0.21068, 0.39041],               Macro: 0.15986 [0.08804, 0.23169]



 86%|████████▌ | 6/7 [02:16<00:39, 39.37s/it]

Test accuracy for 'sentence_structure_featurize': 0.40000, 95% CIs for accuracy: [0.30398 0.49602]
Test F1 scores for 'sentence_structure_featurize':               Weighted: 0.35226 [0.25863, 0.44588],               Macro: 0.22662 [0.14457, 0.30867]



100%|██████████| 7/7 [02:16<00:00, 19.51s/it]

Test accuracy for 'pronoun_featurize': 0.47000, 95% CIs for accuracy: [0.37218 0.56782]
Test F1 scores for 'pronoun_featurize':               Weighted: 0.30054 [0.21068, 0.39041],               Macro: 0.15986 [0.08804, 0.23169]



For the Splits folder, we see that binary_bow_featurize, bigram, appeal_to_fear_featurize, trigram, name_calling_featurize, and pronoun_featurize return the highest accuracy

In [9]:
warnings.filterwarnings("ignore")
trainingFile = "splits_v2/train.txt"
devFile = "splits_v2/dev.txt"
testFile = "splits_v2/test.txt"
    
run(trainingFile, devFile, testFile)


 14%|█▍        | 1/7 [00:11<01:11, 11.99s/it]

Test accuracy for 'binary_bow_featurize': 0.62000, 95% CIs for accuracy: [0.52487 0.71513]
Test F1 scores for 'binary_bow_featurize':               Weighted: 0.60535 [0.50955, 0.70115],               Macro: 0.43804 [0.34080, 0.53529]



 29%|██▊       | 2/7 [00:15<00:35,  7.20s/it]

Test accuracy for 'bigram': 0.54000, 95% CIs for accuracy: [0.44232 0.63768]
Test F1 scores for 'bigram':               Weighted: 0.49323 [0.39524, 0.59122],               Macro: 0.33490 [0.24240, 0.42740]



 43%|████▎     | 3/7 [00:17<00:19,  4.75s/it]

...

Test accuracy for 'name_calling_featurize': 0.47000, 95% CIs for accuracy: [0.37218 0.56782]
Test F1 scores for 'name_calling_featurize':               Weighted: 0.31451 [0.22350, 0.40551],               Macro: 0.21197 [0.13187, 0.29208]



 86%|████████▌ | 6/7 [01:42<00:28, 28.88s/it]

Test accuracy for 'sentence_structure_featurize': 0.40000, 95% CIs for accuracy: [0.30398 0.49602]
Test F1 scores for 'sentence_structure_featurize':               Weighted: 0.42237 [0.32556, 0.51918],               Macro: 0.24731 [0.16275, 0.33187]



100%|██████████| 7/7 [01:42<00:00, 14.67s/it]

Test accuracy for 'pronoun_featurize': 0.47000, 95% CIs for accuracy: [0.37218 0.56782]
Test F1 scores for 'pronoun_featurize':               Weighted: 0.30054 [0.21068, 0.39041],               Macro: 0.15986 [0.08804, 0.23169]



For the Splits_v2 folder, we see that binary_bow_featurize, bigram, appeal_to_fear_featurize, name_calling_featurize, and pronoun_featurize return the highest accuracy

In [10]:
trainingFile = "splits/train.txt"
devFile = "splits/dev.txt"
testFile = "splits/test.txt"
trainX, trainY=load_data(trainingFile)
devX, devY=load_data(devFile)
testX, testY=load_data(testFile)

classifier = Classifier([binary_bow_featurize, bigram, trigram, appeal_to_fear_featurize, name_calling_featurize, pronoun_featurize],
                        trainX, trainY, devX, devY, testX, testY)
classifier.train()
accuracy = classifier.test()
f1_weighted, f1_macro = classifier.f1_test()
lower, upper = confidence_intervals(accuracy, len(testY), .95)
weighted_f1_lower, weighted_f1_upper = confidence_intervals(f1_weighted, len(testY), .95)
macro_f1_lower, macro_f1_upper = confidence_intervals(f1_macro, len(testY), .95)
print(f"Test accuracy for Splits Folder combined features: {accuracy:.5f}, 95% CIs: [{lower:.5f} {upper:.5f}]")
print(f"Test F1 scores for Splits Folder combined features: \
      Weighted: {f1_weighted:.5f} [{weighted_f1_lower:.5f}, {weighted_f1_upper:.5f}], \
      Macro: {f1_macro:.5f} [{macro_f1_lower:.5f}, {macro_f1_upper:.5f}]\n")

classifier2 = Classifier([binary_bow_featurize, bigram, appeal_to_fear_featurize, name_calling_featurize], 
                         trainX, trainY, devX, devY, testX, testY)
classifier2.train()
accuracy = classifier2.test()
f1_weighted, f1_macro = classifier2.f1_test()
lower, upper = confidence_intervals(accuracy, len(testY), .95)
weighted_f1_lower, weighted_f1_upper = confidence_intervals(f1_weighted, len(testY), .95)
macro_f1_lower, macro_f1_upper = confidence_intervals(f1_macro, len(testY), .95)
print(f"Test accuracy for Splits Folder binary bow + bigram + appeal to fear + name calling features: {accuracy:.5f}, \
      95% CIs: [{lower:.5f} {upper:.5f}]")

print(f"Test F1 scores for Splits Folder binary bow + bigram + appeal to fear + name calling features: \
      Weighted: {f1_weighted:.5f} [{weighted_f1_lower:.5f}, {weighted_f1_upper:.5f}], \
      Macro: {f1_macro:.5f} [{macro_f1_lower:.5f}, {macro_f1_upper:.5f}]\n")

Test accuracy for Splits Folder combined features: 0.56000, 95% CIs: [0.46271 0.65729]
Test F1 scores for Splits Folder combined features:       Weighted: 0.53130 [0.43350, 0.62911],       Macro: 0.33774 [0.24504, 0.43043]

Test accuracy for Splits Folder binary bow + bigram + appeal to fear + name calling features: 0.59000,       95% CIs: [0.49360 0.68640]
Test F1 scores for Splits Folder binary bow + bigram + appeal to fear + name calling features:       Weighted: 0.56073 [0.46346, 0.65800],       Macro: 0.37358 [0.27877, 0.46839]



In [11]:
trainingFile = "splits_v2/train.txt"
devFile = "splits_v2/dev.txt"
testFile = "splits_v2/test.txt"
trainX, trainY=load_data(trainingFile)
devX, devY=load_data(devFile)
testX, testY=load_data(testFile)

classifier = Classifier([binary_bow_featurize, bigram, trigram, appeal_to_fear_featurize, name_calling_featurize, pronoun_featurize],
                        trainX, trainY, devX, devY, testX, testY)
classifier.train()
accuracy = classifier.test()
f1_weighted, f1_macro = classifier.f1_test()
lower, upper = confidence_intervals(accuracy, len(testY), .95)
weighted_f1_lower, weighted_f1_upper = confidence_intervals(f1_weighted, len(testY), .95)
macro_f1_lower, macro_f1_upper = confidence_intervals(f1_macro, len(testY), .95)
print(f"Test accuracy for Splits_V2 Folder combined features: {accuracy:.5f}, 95% CIs: [{lower:.5f} {upper:.5f}]")
print(f"Test F1 scores for Splits_V2 Folder combined features: \
      Weighted: {f1_weighted:.5f} [{weighted_f1_lower:.5f}, {weighted_f1_upper:.5f}], \
      Macro: {f1_macro:.5f} [{macro_f1_lower:.5f}, {macro_f1_upper:.5f}]\n")

classifier2 = Classifier([binary_bow_featurize, bigram, appeal_to_fear_featurize, name_calling_featurize], 
                         trainX, trainY, devX, devY, testX, testY)
classifier2.train()
accuracy = classifier2.test()
f1_weighted, f1_macro = classifier2.f1_test()
lower, upper = confidence_intervals(accuracy, len(testY), .95)
weighted_f1_lower, weighted_f1_upper = confidence_intervals(f1_weighted, len(testY), .95)
macro_f1_lower, macro_f1_upper = confidence_intervals(f1_macro, len(testY), .95)

print(f"Test accuracy for Splits_v2 Folder binary bow + bigram + appeal to fear + name calling features: {accuracy:.5f}, \
      95% CIs: [{lower:.5f} {upper:.5f}]")
print(f"Test F1 scores for Splits_v2 Folder binary bow + bigram + appeal to fear + name calling features: \
      Weighted: {f1_weighted:.5f} [{weighted_f1_lower:.5f}, {weighted_f1_upper:.5f}], \
      Macro: {f1_macro:.5f} [{macro_f1_lower:.5f}, {macro_f1_upper:.5f}]\n")

Test accuracy for Splits_V2 Folder combined features: 0.56000, 95% CIs: [0.46271 0.65729]
Test F1 scores for Splits_V2 Folder combined features:       Weighted: 0.55126 [0.45377, 0.64874],       Macro: 0.43593 [0.33874, 0.53312]

Test accuracy for Splits_v2 Folder binary bow + bigram + appeal to fear + name calling features: 0.65000,       95% CIs: [0.55652 0.74348]
Test F1 scores for Splits_v2 Folder binary bow + bigram + appeal to fear + name calling features:       Weighted: 0.63912 [0.54499, 0.73325],       Macro: 0.54598 [0.44840, 0.64357]



## Summary of Work and Results

In [ ]:
print(f"Name Calling Count: {Y.count('Name Calling')} \n \
      Appeal to Fear Count: {Y.count('Appeal to Fear')} \n \
      No Propaganda Count: {Y.count('No Propaganda')} \n \
      Loaded Language Count: {Y.count('Loaded Language')}")

Name Calling Count: 28 
 Appeal to Fear Count: 61 
 No Propaganda Count: 235 
 Loaded Language Count: 176


The first thing to note is that there is severe class imbalance within our dataset where 47% is No Propaganda, 35.2% is Loaded Language, 12.2% is Appeal to Fear, and 5.6% is Name Calling. To counteract this class imbalance, [SMOTE](https://arxiv.org/pdf/1106.1813) was employed to then have equal samples for training. Prior to SMOTE, the Logistic Regression gave an accuracy of 47% with binary_bow_featurize and bigram as features. With SMOTE, the Logistic Regression gave an accuracy of 50% with binary_bow_featurize and bigram as features. 

In addition, I attempted using varying classification models and from prior experience felt that OneVsRest Classifiers were effective for trying to get a singular classification from multiple options. Both LinearSVC and ComplementNB were tested to see if they were effective. ComplementNB with SMOTE achieved an accuracy of 55% with binary_bow_featurize and bigram as features. LinearSVC with SMOTE achieved an accuracy of 56.1% with binary_bow_featurize and bigram as features.

This then led me to think of useful features to implement. I first tried trigrams and quadrigrams but got diminishing returns on accuracy. I then looked at the unique words that occurred more often in the minority classes to give more prediction power for these classes to the model. I also created a feature for the length of each text document to see if that would be beneficial to the model. I also hypothesized that certain classes might have higher amounts of third-person language to convey disappointment or fear to the read which is why I created the pronoun_featurize function.

There was also some confusion within the group due to inherent formatting issues with the data which is why there is splits and splits_v2 as different renditions of the splitting method, but it still adheres to the guidelines posted. No matter the split folder, the model outperforms the MajorityClassifier accuracy of 47%. The splits folder relays 59% accuracy while the splits_v2 folder relays 65% accuracy.

As a holistic note, due to the large class imbalance, accuracy may not be the best metric, but rather Macro or Weighted F1 score which is why those are included.